# Django 4 — OWASP Secure Coding

## Prerequisites
- Python 3.10+
- Django 4.x
- Basic knowledge of Django views, models, and forms
- Familiarity with HTTP and web application concepts

## Learning Objectives
By the end of this module you will be able to:
- Explain what OWASP is and how ASVS v4.0.3 structures security requirements
- Implement secure authentication with Django's built-in auth backend, rate limiting, and two-factor authentication
- Manage sessions securely using HTTPS-only cookies and proper session lifecycle controls
- Enforce access control through roles, claims, and CSRF protection
- Validate, sanitise, and encode user input to prevent XSS, SQL injection, and OS injection
- Protect sensitive data at rest and in transit using encryption and TLS
- Configure logging to capture security events without leaking sensitive information
- Harden dependencies, disable debug in production, and apply security headers
- Secure business logic flows against automation attacks and workflow bypasses
- Handle file uploads safely and prevent SSRF attacks
- Secure REST APIs with authentication, schema validation, and CORS

## Table of Contents
1. [Introduction to OWASP and ASVS](#1-introduction-to-owasp-and-asvs)
2. [Authentication (ASVS Ch.2)](#2-authentication-asvs-ch2)
3. [Session Management (ASVS Ch.3)](#3-session-management-asvs-ch3)
4. [Access Control (ASVS Ch.4)](#4-access-control-asvs-ch4)
5. [Validation, Sanitisation and Encoding (ASVS Ch.5)](#5-validation-sanitisation-and-encoding-asvs-ch5)
6. [Protecting Sensitive Data (ASVS Ch.6, 8, 9)](#6-protecting-sensitive-data-asvs-ch6-8-9)
7. [Error Handling and Logging (ASVS Ch.7)](#7-error-handling-and-logging-asvs-ch7)
8. [Hardening Configuration (ASVS Ch.10, 14)](#8-hardening-configuration-asvs-ch10-14)
9. [Business Logic Security (ASVS Ch.11)](#9-business-logic-security-asvs-ch11)
10. [Securing Third-party Files (ASVS Ch.12)](#10-securing-third-party-files-asvs-ch12)
11. [API and Web Services (ASVS Ch.13)](#11-api-and-web-services-asvs-ch13)

```python
from pathlib import Path
from IPython.display import Image, display

def render_diagram(image_path, width=900):
    display(Image(filename=str(image_path), width=width))

print("Setup complete. render_diagram() is ready.")
```

---
## 1. Introduction to OWASP and ASVS

### What Is OWASP?

**OWASP** (Open Web Application Security Project) is an open-source, non-profit foundation that produces freely available security tools, standards, and guidelines.

Key OWASP projects:
- **OWASP Top 10** — the ten most critical web application security risks
- **OWASP Testing Guidelines** — a comprehensive manual for testing web applications
- **OWASP Cheatsheets** — quick-reference guides for common security topics
- **OWASP ASVS** — Application Security Verification Standard (what this module focuses on)

### Why Do We Need Security Standards?

- Security must be considered at **every stage** of the SDLC, not bolted on at the end.
- A standard gives teams **clear trust boundaries** and **measurable requirements**.
- Using a well-known standard prevents reinventing the wheel and ensures comprehensive coverage.

### OWASP ASVS v4.0.3

The **Application Security Verification Standard** defines security requirements that can be used to verify (test) an application's security posture. It covers 14 chapters, each mapping to a specific security domain.

### ASVS Verification Levels

ASVS defines three verification levels:

| Level | Description | Use Case |
|-------|-------------|----------|
| **Level 1** | Bare minimum; fully testable through penetration testing | Any application — the baseline |
| **Level 2** | Standard for applications processing sensitive data | Payment systems, healthcare, HR apps |
| **Level 3** | Highest assurance; requires design review and threat modelling | Military, high-value targets, critical infrastructure |

- Levels are **additive**: Level 2 includes all Level 1 requirements, Level 3 includes all of Level 2.
- This course focuses on **Level 1** — the requirements every Django application should meet.
- ASVS requirements are tagged by level so teams can plan incremental adoption.

### Secure SDLC — Security at Every Stage

Security must be built in, not bolted on. The SDLC stages and their security activities:

| Stage | Security Activity |
|-------|------------------|
| **Requirements** | Threat modelling, identifying security requirements, choosing ASVS level |
| **Design** | Defining trust boundaries, selecting secure patterns, architecture review |
| **Code** | Secure coding standards, peer review, static analysis (SAST) |
| **Test** | Penetration testing, DAST, dependency scanning |
| **Deploy** | Environment hardening, secrets management, disabling debug |
| **Operate** | Monitoring, incident response, regular re-verification |

### ASVS Security Architecture Domains

The standard is structured around 14 chapters, each covering a security domain:

| Chapter | Domain | Key Concern |
|---------|--------|-------------|
| **Ch.2** | Authentication | Passwords, MFA, credential recovery |
| **Ch.3** | Session Management | Token generation, cookie security, lifecycle |
| **Ch.4** | Access Control | Roles, claims, CSRF |
| **Ch.5** | Validation, Sanitisation & Encoding | XSS, injection, deserialisation |
| **Ch.6** | Cryptography | Algorithms, key management |
| **Ch.7** | Error Handling & Logging | What to log, what not to leak |
| **Ch.8** | Data Protection | PII, privacy, data minimisation |
| **Ch.9** | Communication Security | TLS, certificate validation, HSTS |
| **Ch.10** | Malicious Code | Dependency integrity, backdoors |
| **Ch.11** | Business Logic | Workflow enforcement, anti-automation |
| **Ch.12** | Files & Resources | Upload safety, SSRF |
| **Ch.13** | API & Web Services | RESTful security, CORS, JSON schema |
| **Ch.14** | Configuration | Debug flags, security headers, server hardening |

### Configuration Architecture

- **Segregation of components in production** — isolate services, databases, and infrastructure from each other
- **Secure build and deploy**
  - Check outdated dependencies before every release
  - Verify package signatures / integrity hashes
- **Deprecate support for outdated technology** — remove legacy TLS versions, deprecated cipher suites, end-of-life libraries

---
## 2. Authentication (ASVS Ch.2)

Authentication verifies that users are who they claim to be. ASVS Chapter 2 defines requirements across several subsections.

render_diagram("assets/ch2_authentication_sections.png")

```python
# Django password validator configuration (settings.py)
AUTH_PASSWORD_VALIDATORS_EXAMPLE = [
    {
        'NAME': 'django.contrib.auth.password_validation.UserAttributeSimilarityValidator',
    },
    {
        'NAME': 'django.contrib.auth.password_validation.MinimumLengthValidator',
        'OPTIONS': {'min_length': 12},  # ASVS 2.1.1: minimum 12 characters
    },
    {
        'NAME': 'django.contrib.auth.password_validation.CommonPasswordValidator',
    },
    {
        'NAME': 'django.contrib.auth.password_validation.NumericPasswordValidator',
    },
]

print("AUTH_PASSWORD_VALIDATORS configured with minimum length of 12 characters.")
```

### Rate Limiting with `django-ratelimit`

ASVS 2.2.1 requires that brute-force attacks against credentials are detected and blocked. Django does not include rate limiting by default — use `django-ratelimit`.

### Two-Factor Authentication with `django-otp`

ASVS 2.6 encourages support for TOTP-based 2FA. `django-otp` integrates cleanly with Django's auth system.

```python
# Install: pip install django-otp

# settings.py additions
INSTALLED_APPS_ADDITIONS = [
    'django_otp',
    'django_otp.plugins.otp_totp',  # time-based OTP
    'django_otp.plugins.otp_static',  # backup codes
]

MIDDLEWARE_ADDITIONS = [
    'django_otp.middleware.OTPMiddleware',  # must come after AuthenticationMiddleware
]

# views.py — require verified OTP before granting access
from django_otp.decorators import otp_required
from django.contrib.auth.decorators import login_required

@login_required
@otp_required
def sensitive_view(request):
    # Only reachable if user is authenticated AND has passed 2FA
    return render(request, 'sensitive.html')

print("django-otp configuration shown. Use @otp_required for sensitive views.")
```

### Password Reset Security

ASVS 2.5 governs credential recovery. Django's built-in `PasswordResetView` satisfies most requirements, but verify:
- Token expiry is short (default `PASSWORD_RESET_TIMEOUT = 3600` seconds — 1 hour; consider reducing to 900)
- Tokens are single-use (Django's built-in implementation already enforces this via HMAC)
- Reset emails never reveal whether an account exists (use a consistent message like "If this email exists…")

```python
# settings.py — shorten token expiry to 15 minutes (ASVS 2.5.3)
PASSWORD_RESET_TIMEOUT = 900  # 15 minutes in seconds

print(f"PASSWORD_RESET_TIMEOUT set to {PASSWORD_RESET_TIMEOUT} seconds (15 minutes).")
```

---
## 3. Session Management (ASVS Ch.3)

Session management controls how a user's authenticated state is maintained across HTTP requests.

```python
render_diagram("assets/ch3_session_management_sections.png")
```

### ASVS Ch.3 Key Requirements

| Section | Requirement |
|---------|-------------|
| **3.2** | Session tokens must be at least 128 bits; use cryptographically random generation |
| **3.3** | Sessions expire after logout and after inactivity |
| **3.4** | Cookie attributes: `HttpOnly`, `Secure`, `SameSite` |
| **3.7** | Step-up re-authentication for privileged actions |

### Secure Cookie Configuration in Django

```python
# settings.py — secure session and CSRF cookie settings

# Session cookie settings (ASVS 3.4)
SESSION_COOKIE_HTTPONLY = True    # Prevents JavaScript access
SESSION_COOKIE_SECURE = True      # Only sent over HTTPS
SESSION_COOKIE_SAMESITE = 'Lax'   # Protects against CSRF; use 'Strict' for stricter sites
SESSION_COOKIE_AGE = 1800         # Idle timeout: 30 minutes
SESSION_SAVE_EVERY_REQUEST = True # Sliding expiry: resets timer on every request
SESSION_EXPIRE_AT_BROWSER_CLOSE = True  # Session dies when browser closes

# CSRF cookie settings
CSRF_COOKIE_HTTPONLY = True
CSRF_COOKIE_SECURE = True
CSRF_COOKIE_SAMESITE = 'Lax'

print("Secure session and CSRF cookie settings configured.")
```

### Session Binding and Invalidation

After a user logs in, Django should generate a **new** session ID to prevent session fixation attacks. Django's `login()` function calls `request.session.cycle_key()` automatically — but only if a session already exists.

On logout, the session must be destroyed server-side:

```python
from django.contrib.auth import logout as auth_logout
from django.shortcuts import redirect

def logout_view(request):
    auth_logout(request)       # Django flushes the session server-side
    response = redirect('login')
    response.delete_cookie('sessionid')  # Belt-and-suspenders: clear client-side too
    return response

print("Secure logout: server-side session flush + client-side cookie deletion.")
```

### Step-Up Authentication (ASVS 3.7)

For sensitive operations (changing email, payment), re-confirm identity even if the user already has a valid session:

```python
import time
from django.contrib.auth import authenticate
from django.shortcuts import render

STEP_UP_VALIDITY_SECONDS = 300  # Step-up lasts 5 minutes

def change_email_view(request):
    # Check if user has recently re-authenticated
    last_auth = request.session.get('step_up_auth_at', 0)
    if time.time() - last_auth > STEP_UP_VALIDITY_SECONDS:
        return redirect('step_up_auth')
    # ... proceed with sensitive operation
    return render(request, 'change_email.html')

def step_up_auth_view(request):
    if request.method == 'POST':
        password = request.POST['password']
        user = authenticate(request, username=request.user.username, password=password)
        if user:
            request.session['step_up_auth_at'] = time.time()
            return redirect('change_email')
    return render(request, 'step_up.html')

print("Step-up authentication pattern: re-confirm password for sensitive operations.")
```

---
## 4. Access Control (ASVS Ch.4)

Access control ensures that authenticated users can only perform actions and access resources they are authorised for.

```python
render_diagram("assets/access_control_definition.png")
```

```python
render_diagram("assets/ch4_access_control_sections.png")
```

### ASVS Ch.4 Key Requirements

| Section | Requirement |
|---------|-------------|
| **4.1** | Verify principle of least privilege; deny by default |
| **4.2** | Operation-level access control; prevent IDOR |
| **4.3** | Admin interfaces require additional authentication; no anonymous access |

### Roles and Permissions in Django

Django's permission framework (`django.contrib.auth`) provides:
- Per-model `add`, `change`, `delete`, `view` permissions
- `Groups` for bundling permissions
- `@permission_required` and `@login_required` decorators
- `user.has_perm('app.permission_name')`

```python
from django.contrib.auth.decorators import login_required, permission_required
from django.core.exceptions import PermissionDenied

# Pattern 1: decorator-based permission check
@login_required
@permission_required('myapp.change_order', raise_exception=True)
def edit_order(request, order_id):
    order = Order.objects.get(pk=order_id)
    # ASVS 4.2.1: also verify object-level ownership
    if order.owner != request.user:
        raise PermissionDenied
    return render(request, 'edit_order.html', {'order': order})

# Pattern 2: class-based view with LoginRequiredMixin
from django.contrib.auth.mixins import LoginRequiredMixin, PermissionRequiredMixin
from django.views.generic import UpdateView

class EditOrderView(LoginRequiredMixin, PermissionRequiredMixin, UpdateView):
    permission_required = 'myapp.change_order'
    model = 'Order'

    def get_queryset(self):
        # Restrict queryset to objects owned by the current user (prevents IDOR)
        return super().get_queryset().filter(owner=self.request.user)

print("Access control patterns: decorator + CBV with queryset scoping.")
```

### CSRF Attacks and Prevention

**CSRF (Cross-Site Request Forgery)** tricks an authenticated user's browser into making an unwanted request to a site where they are logged in. The attacker does not need to see the response — they just need the action to happen.

```python
render_diagram("assets/csrf_prevention_flow.png")
```

**How CSRF works:**
1. User is logged in to `bank.com`.
2. Attacker lures user to `evil.com` which contains `<img src="bank.com/transfer?to=attacker&amount=1000">`.
3. Browser automatically sends the user's `bank.com` session cookie with the request.
4. Bank executes the transfer because it looks like a legitimate request.

**Django's CSRF protection:**
- `CsrfViewMiddleware` is enabled by default.
- Every POST form must include `{% csrf_token %}`.
- The middleware validates the token on every unsafe HTTP method (POST, PUT, PATCH, DELETE).
- For AJAX, include the token in the `X-CSRFToken` header.

```python
# Django template — always include {% csrf_token %} in forms
TEMPLATE_EXAMPLE = """
<form method="post" action="/transfer/">
    {% csrf_token %}
    <input type="number" name="amount">
    <button type="submit">Transfer</button>
</form>
"""

# AJAX requests — include the CSRF token in the header
AJAX_EXAMPLE = """
function getCookie(name) {
    // Django sets the CSRF token in a cookie named 'csrftoken'
    let value = `; ${document.cookie}`;
    let parts = value.split(`; ${name}=`);
    if (parts.length === 2) return parts.pop().split(';').shift();
}

fetch('/transfer/', {
    method: 'POST',
    headers: {
        'X-CSRFToken': getCookie('csrftoken'),
        'Content-Type': 'application/json',
    },
    body: JSON.stringify({ amount: 100 }),
});
"""

print("CSRF protection: use {% csrf_token %} in templates; X-CSRFToken header for AJAX.")
```

---
## 5. Validation, Sanitisation and Encoding (ASVS Ch.5)

Never trust user-supplied data. Every piece of input that enters the application must be validated and, depending on where it's used, sanitised or encoded.

```python
render_diagram("assets/user_data_trust_issues.png")
```

```python
render_diagram("assets/ch5_validation_sections.png")
```

### ASVS Ch.5 Key Requirements

| Section | Requirement |
|---------|-------------|
| **5.1** | All input validated on the server; whitelist where possible |
| **5.2** | Sanitise or encode output to prevent XSS |
| **5.3** | Use parameterised queries; never concatenate SQL |
| **5.4** | Prevent OS command injection; avoid `os.system()` with user input |
| **5.5** | Prevent insecure deserialisation; avoid pickle with untrusted data |

### XSS Prevention

**Cross-Site Scripting (XSS)** injects malicious scripts into web pages viewed by other users. Django's template engine **auto-escapes** HTML by default.

```python
# Django templates auto-escape HTML — {{ variable }} is safe
# Only use |safe or mark_safe() for trusted, controlled content

from django.utils.safestring import mark_safe
from django.utils.html import escape

# DANGEROUS — never do this with user input:
# html = mark_safe(f"<p>Hello {user_input}</p>")

# SAFE — auto-escaped in templates:
# {{ user_input }}  → escapes &, <, >, ", '

# SAFE — manual escaping in Python code:
def safe_greeting(name: str) -> str:
    return escape(name)  # escapes HTML characters

print(safe_greeting("<script>alert('xss')</script>"))
# Output: &lt;script&gt;alert(&#x27;xss&#x27;)&lt;/script&gt;
```

### SQL Injection Prevention

Django's ORM uses parameterised queries by default — but raw SQL is still possible and dangerous.

```python
# DANGEROUS — raw SQL with string formatting
# user_id = request.GET['id']
# User.objects.raw(f"SELECT * FROM auth_user WHERE id = {user_id}")  # NEVER DO THIS

# SAFE — Django ORM (parameterised automatically)
from django.contrib.auth.models import User

def get_user_safe(user_id):
    return User.objects.get(pk=user_id)  # ORM builds parameterised query

# SAFE — if raw SQL is necessary, use parameter binding
def search_users_safe(search_term):
    return User.objects.raw(
        "SELECT * FROM auth_user WHERE username = %s",
        [search_term]  # Passed as bound parameter, never interpolated
    )

print("Use ORM queryset methods; if raw SQL is required, always use parameterised queries.")
```

### OS Command Injection Prevention

Never pass user input to shell commands. Use Python's `subprocess` with a list of arguments (not a string), which prevents shell expansion.

```python
import subprocess

# DANGEROUS — shell=True with user input
# filename = request.POST['filename']
# subprocess.run(f"cat {filename}", shell=True)  # user can inject: ; rm -rf /

# SAFE — list form without shell=True
def read_file_safe(filename: str) -> bytes:
    # Validate first — allow only alphanumeric and dots
    import re
    if not re.match(r'^[a-zA-Z0-9._-]+$', filename):
        raise ValueError("Invalid filename")
    result = subprocess.run(['cat', filename], capture_output=True, timeout=5)
    return result.stdout

print("Prefer Python's file I/O over shell commands. If subprocess is needed, use list form.")
```

### Insecure Deserialisation (ASVS 5.5)

Deserialising untrusted data can lead to remote code execution. Avoid formats with embedded type information.

```python
import json

# DANGEROUS — pickle with untrusted data allows arbitrary code execution
# import pickle
# data = pickle.loads(user_supplied_bytes)  # NEVER with untrusted input

# DANGEROUS — PyYAML full_load with untrusted YAML
# import yaml
# data = yaml.full_load(user_yaml)  # allows Python object construction

# SAFE — JSON only carries data, not code
def safe_deserialise(raw: str) -> dict:
    return json.loads(raw)  # JSON cannot execute code

# SAFE — If YAML is required, use safe_load
# import yaml
# data = yaml.safe_load(user_yaml)  # restricts to basic types only

print("Use JSON for untrusted data. Never use pickle or yaml.full_load with user input.")
```

### Input Validation with Django Forms

```python
from django import forms

class TransferForm(forms.Form):
    amount = forms.DecimalField(
        min_value=0.01,
        max_value=10000.00,
        decimal_places=2,
        max_digits=8,
    )
    recipient_iban = forms.CharField(
        max_length=34,
        # Custom validator
    )

    def clean_recipient_iban(self):
        iban = self.cleaned_data['recipient_iban']
        # Whitelist: IBAN format GB82WEST12345698765432
        import re
        if not re.match(r'^[A-Z]{2}[0-9]{2}[A-Z0-9]{4,30}$', iban):
            raise forms.ValidationError('Invalid IBAN format')
        return iban

print("Django forms provide server-side validation. Always use form.cleaned_data, never request.POST directly.")
```

---
## 6. Protecting Sensitive Data (ASVS Ch.6, 8, 9)

Sensitive data must be protected both at rest (stored in databases/files) and in transit (over the network).

```python
render_diagram("assets/data_lifecycle.png")
```

```python
render_diagram("assets/ch689_asvs_requirements.png")
```

### ASVS Ch.6 / 8 / 9 Key Requirements

| Chapter | Topic | Key Requirements |
|---------|-------|------------------|
| **6** | Cryptography | AES-256 for symmetric; RSA 2048+ or ECC P-256+ for asymmetric; use random IVs |
| **8** | Data Protection | Classify sensitive data; minimal storage; signed cookies; data expiry |
| **9** | Communication | TLS 1.2+ only; HSTS; no mixed content; validate server certificates |

### Classifying Sensitive Data

Before protecting data, identify what is sensitive:
- **Personally Identifiable Information (PII)**: name, email, address, date of birth
- **Credentials**: passwords, API keys, tokens
- **Financial data**: credit card numbers, bank accounts, transaction history
- **Health data**: medical records, prescriptions

### Data Protection at Rest

```python
# Django signed cookies — data is signed (not encrypted) to prevent tampering
from django.core import signing

def set_signed_cookie(response, key, value):
    signer = signing.Signer()
    response.set_cookie(key, signer.sign(value))

def read_signed_cookie(request, key):
    signer = signing.Signer()
    try:
        return signer.unsign(request.COOKIES.get(key, ''))
    except signing.BadSignature:
        return None  # tampered cookie

# For field-level encryption in the database, use django-fernet-fields or similar
# pip install django-fernet-fields
# from fernet_fields import EncryptedCharField
# class UserProfile(models.Model):
#     national_id = EncryptedCharField(max_length=20)  # encrypted at rest

print("Use Django signed cookies for tamper-evident data. Use field encryption for sensitive DB columns.")
```

### TLS and HTTPS Configuration

```python
# settings.py — enforce HTTPS in production

# Redirect all HTTP requests to HTTPS
SECURE_SSL_REDIRECT = True

# HTTP Strict Transport Security (HSTS) — tells browsers to always use HTTPS
SECURE_HSTS_SECONDS = 31536000       # 1 year
SECURE_HSTS_INCLUDE_SUBDOMAINS = True
SECURE_HSTS_PRELOAD = True           # submit to browsers' preload list

# Trust the X-Forwarded-Proto header from the load balancer
SECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')

print("HTTPS enforcement: SECURE_SSL_REDIRECT + HSTS configured.")
```

### Privacy and User Consent (ASVS 8.3.3)

ASVS 8.3.3 requires that users can request deletion of their personal data. GDPR also mandates a "right to be forgotten". Design your data model with privacy in mind:
- Store only what you need (data minimisation)
- Set retention periods and delete stale data automatically
- Provide a data deletion endpoint for users

```python
from django.contrib.auth.decorators import login_required
from django.http import JsonResponse

@login_required
def delete_my_data(request):
    if request.method == 'POST':
        user = request.user
        # Anonymise or delete PII
        user.email = f'deleted_{user.pk}@example.com'
        user.first_name = ''
        user.last_name = ''
        user.save()
        user.profile.delete()  # delete related PII
        auth_logout(request)
        return JsonResponse({'status': 'data deleted'})
    return JsonResponse({'error': 'POST required'}, status=405)

print("GDPR right-to-be-forgotten: provide a data deletion endpoint.")
```

---
## 7. Error Handling and Logging (ASVS Ch.7)

Proper logging helps diagnose problems and detect security incidents. Improper logging leaks sensitive data.

```python
render_diagram("assets/logging_in_app.png")
```

```python
render_diagram("assets/ch7_asvs_requirements.png")
```

### ASVS Ch.7 Key Requirements

| Section | Requirement |
|---------|-------------|
| **7.1** | Do not log: passwords, payment details, session tokens, PII |
| **7.2** | Log: security events (login success/failure, permission violations, data export) |
| **7.4** | Display generic errors to users; log detailed errors server-side only |

### Logging Configuration in Django

Django uses Python's standard `logging` library. Configure it in `settings.py`.

```python
# settings.py — logging configuration
import logging

LOGGING = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'verbose': {
            'format': '{levelname} {asctime} {module} {message}',
            'style': '{',
        },
        'security': {
            'format': 'SECURITY {asctime} user={user} action={action} ip={ip} {message}',
            'style': '{',
        },
    },
    'filters': {
        'sensitive_filter': {
            '()': 'myapp.logging_filters.SensitiveDataFilter',  # custom filter
        },
    },
    'handlers': {
        'file': {
            'class': 'logging.handlers.RotatingFileHandler',
            'filename': '/var/log/myapp/app.log',
            'maxBytes': 10 * 1024 * 1024,  # 10 MB
            'backupCount': 5,
            'formatter': 'verbose',
        },
        'security_file': {
            'class': 'logging.handlers.RotatingFileHandler',
            'filename': '/var/log/myapp/security.log',
            'formatter': 'verbose',
        },
    },
    'loggers': {
        'django': {
            'handlers': ['file'],
            'level': 'WARNING',
        },
        'myapp.security': {
            'handlers': ['security_file'],
            'level': 'INFO',
            'propagate': False,
        },
    },
}

print("Logging configured with separate security audit log.")
```

### Obfuscating Sensitive Data in Logs

```python
import logging
import re

class SensitiveDataFilter(logging.Filter):
    """Remove sensitive fields from log records."""

    SENSITIVE_KEYS = ['password', 'token', 'credit_card', 'cvv', 'secret']

    def filter(self, record: logging.LogRecord) -> bool:
        if isinstance(record.msg, str):
            for key in self.SENSITIVE_KEYS:
                pattern = rf'(?i)({key}["\']?\s*[:=]\s*)["\']?[^"\',}}\s]+'
                record.msg = re.sub(pattern, r'\1[REDACTED]', record.msg)
        return True  # always allow the record; only modify content

# Usage
logger = logging.getLogger('myapp.security')

def log_login_attempt(username: str, success: bool, ip: str):
    status = 'SUCCESS' if success else 'FAILURE'
    # NOTE: never log the password, only the username and outcome
    logger.info(f"LOGIN_{status} username={username} ip={ip}")

print("SensitiveDataFilter redacts passwords, tokens, and credit card data from logs.")
```

### Generic Error Messages (ASVS 7.4.1)

Show generic errors to users; log details server-side. Never expose stack traces, internal paths, or SQL errors in production.

```python
# settings.py
DEBUG = False  # ASVS 14.3.2: never enable DEBUG in production

# Custom error handlers in urls.py
# handler404 = 'myapp.views.error_404'
# handler500 = 'myapp.views.error_500'

# views.py
import logging

logger = logging.getLogger(__name__)

def error_500(request):
    # Log details internally
    logger.error("Internal Server Error", exc_info=True, extra={'request': request})
    # Return a generic message to the user
    return render(request, '500.html', status=500)

print("Generic errors to users; detailed logging server-side. DEBUG=False in production.")
```

---
## 8. Hardening Configuration (ASVS Ch.10, 14)

Configuration hardening protects against malicious dependencies and reduces the attack surface of a deployed application.

```python
render_diagram("assets/malicious_code_checks.png")
```

```python
render_diagram("assets/ch10_14_asvs_requirements.png")
```

### ASVS Ch.10 and Ch.14 Key Requirements

| Section | Requirement |
|---------|-------------|
| **10.3.1** | Verify package integrity with hash verification |
| **10.3.2** | Protect external resources with Subresource Integrity (SRI) |
| **14.2.1** | Keep all dependencies up-to-date; scan for known vulnerabilities |
| **14.3.2** | Disable debug features in production |
| **14.3.3** | Do not expose server version in HTTP headers |
| **14.4** | Add security HTTP headers |
| **14.5.1** | Validate HTTP methods; reject unexpected ones |

### Dependency Security

`pip` by default does not verify package integrity. Use `pip-tools` and hash verification to protect against tampered packages.

```python
# Generate requirements with hashes (ASVS 10.3.1)
# pip-compile --generate-hashes requirements.in > requirements.txt

# requirements.txt with hashes (example excerpt)
REQUIREMENTS_EXAMPLE = """
Django==4.2.7 \\
    --hash=sha256:8e0f1c2a... \\
    --hash=sha256:a1b2c3d4...
"""

# Install with hash checking
# pip install --require-hashes -r requirements.txt

# Audit for known vulnerabilities with 'safety'
# pip install safety
# safety check -r requirements.txt

# Check for outdated packages
# pip list --outdated

print("Use pip-compile with --generate-hashes and run 'safety check' in CI.")
```

### Subresource Integrity for JS/CSS (ASVS 10.3.2)

```python
# Always add integrity and crossorigin attributes when loading scripts from CDNs
SRI_EXAMPLE = """
<script
    src="https://cdn.example.com/bootstrap.min.js"
    integrity="sha384-abc123..."
    crossorigin="anonymous">
</script>
"""

# Generate the SRI hash:
# openssl dgst -sha384 -binary bootstrap.min.js | openssl base64 -A

print("SRI: browsers verify the script hash before executing it.")
```

### Security HTTP Headers (ASVS 14.4)

```python
# settings.py — security headers (also see django-csp for Content-Security-Policy)

# Prevent browsers from MIME-sniffing the content type
SECURE_CONTENT_TYPE_NOSNIFF = True

# Enable browser XSS filter (legacy, but still good to have)
SECURE_BROWSER_XSS_FILTER = True

# Clickjacking protection: disallow embedding in iframes
X_FRAME_OPTIONS = 'DENY'

# Content Security Policy (requires django-csp)
# pip install django-csp
CSP_DEFAULT_SRC_EXAMPLE = ("'self'",)
CSP_SCRIPT_SRC_EXAMPLE = ("'self'", "https://cdn.example.com")
CSP_STYLE_SRC_EXAMPLE = ("'self'", "'unsafe-inline'")

# Referrer Policy
# SECURE_REFERRER_POLICY = 'strict-origin-when-cross-origin'

print("Security headers: NOSNIFF, XSS filter, DENY framing, CSP configured.")
```

### Production Django Settings Checklist

```python
# Production security checklist
# Run: python manage.py check --deploy

PRODUCTION_SETTINGS = {
    'DEBUG': False,                          # ASVS 14.3.2
    'SECRET_KEY': '<from-environment-var>',  # Never hardcode
    'ALLOWED_HOSTS': ['myapp.example.com'],  # Restrict to production domain
    'SECURE_SSL_REDIRECT': True,
    'SESSION_COOKIE_SECURE': True,
    'CSRF_COOKIE_SECURE': True,
    'SECURE_HSTS_SECONDS': 31536000,
    'SECURE_HSTS_INCLUDE_SUBDOMAINS': True,
    'SECURE_CONTENT_TYPE_NOSNIFF': True,
    'X_FRAME_OPTIONS': 'DENY',
}

print("Run `python manage.py check --deploy` to validate production settings.")
```

---
## 9. Business Logic Security (ASVS Ch.11)

Business logic vulnerabilities arise when attackers exploit the intended functionality of an application in unintended ways.

```python
render_diagram("assets/business_logic_attacks.png")
```

```python
render_diagram("assets/ch11_asvs_requirements.png")
```

### ASVS Ch.11 Key Requirements

| Section | Requirement |
|---------|-------------|
| **11.1.1** | Verify sequential workflow steps — no skipping allowed |
| **11.1.2** | Process transactions realistically — prevent double-spending |
| **11.1.3** | Rate limit business actions to prevent automation abuse |
| **11.1.4** | Protect against automation attacks (bots, scrapers) |
| **11.1.5** | Validate all inputs against business rules and expected threat models |

### Types of Business Logic Attacks

- **Circumventing workflows**: jumping from step 1 to step 5 in a checkout flow
- **Abusing limits**: buying more than the allowed quantity per user
- **Privilege escalation through inputs**: modifying a hidden field to change the price

### Enforcing Workflow Order (ASVS 11.1.1)

```python
# Enforce multi-step workflow order via session state

CHECKOUT_STEPS = ['cart', 'address', 'payment', 'confirmation']

def checkout_payment(request):
    # Verify the user has completed the previous step
    completed_steps = request.session.get('checkout_steps', [])
    if 'address' not in completed_steps:
        # User tried to skip the address step
        return redirect('checkout_address')
    
    # Process payment...
    completed_steps.append('payment')
    request.session['checkout_steps'] = completed_steps
    return render(request, 'checkout/payment.html')

print("Workflow enforcement: validate that prior steps are complete before allowing next step.")
```

### Rate Limiting Business Actions (ASVS 11.1.3)

Some actions should be limited per user, not just per IP, to prevent automated abuse.

```python
from django_ratelimit.decorators import ratelimit

# Rate limit discount code application to 10 per hour per user
@login_required
@ratelimit(key='user', rate='10/h', block=True)
def apply_discount_code(request):
    if request.method == 'POST':
        code = request.POST.get('code', '')
        # Validate and apply discount...
        pass
    return render(request, 'apply_discount.html')

# Rate limit purchase actions to prevent automated bulk buying
@login_required
@ratelimit(key='user', rate='5/m', block=True)
def purchase_ticket(request):
    pass

print("Use key='user' for per-user rate limiting of business actions.")
```

---
## 10. Securing Third-party Files (ASVS Ch.12)

File uploads are a common attack vector. Untrusted files can harm the application in three main ways.

```python
render_diagram("assets/untrusted_file_harms.png")
```

```python
render_diagram("assets/ch12_asvs_requirements.png")
```

### ASVS Ch.12 Key Requirements

| Section | Requirement |
|---------|-------------|
| **12.1** | Limit file upload size (prevent DoS) |
| **12.3** | Validate filenames and extensions; prevent path traversal |
| **12.4** | Scan uploaded files for malware; store outside webroot |
| **12.5** | Use allowlist for downloadable file types |
| **12.6** | Prevent SSRF with domain allowlisting |

### Secure File Upload in Django

```python
import os
import uuid
import re
from django.core.exceptions import ValidationError
from django.conf import settings

# Allowlisted file extensions
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.gif', '.pdf'}
MAX_UPLOAD_SIZE = 5 * 1024 * 1024  # 5 MB

def validate_uploaded_file(uploaded_file):
    # 1. Check size (ASVS 12.1)
    if uploaded_file.size > MAX_UPLOAD_SIZE:
        raise ValidationError("File too large (max 5 MB).")

    # 2. Validate extension allowlist (ASVS 12.3)
    _, ext = os.path.splitext(uploaded_file.name.lower())
    if ext not in ALLOWED_EXTENSIONS:
        raise ValidationError(f"File type {ext!r} not allowed.")

    # 3. Prevent path traversal — strip directory components (ASVS 12.3)
    if re.search(r'[/\\]', uploaded_file.name):
        raise ValidationError("Invalid filename.")

def save_uploaded_file(uploaded_file):
    validate_uploaded_file(uploaded_file)
    # 4. Randomise the stored filename to prevent location disclosure (ASVS 12.4)
    _, ext = os.path.splitext(uploaded_file.name.lower())
    safe_name = f"{uuid.uuid4().hex}{ext}"
    # 5. Store outside MEDIA_ROOT / webroot — serve via a view, not directly
    upload_path = os.path.join(settings.SECURE_UPLOAD_DIR, safe_name)
    with open(upload_path, 'wb') as f:
        for chunk in uploaded_file.chunks():
            f.write(chunk)
    return safe_name

print("Secure upload: size check, extension allowlist, randomised filename, external storage.")
```

### Virus Scanning with pyclamd (ASVS 12.4.2)

```python
# pip install pyclamd
# Requires ClamAV daemon to be running: apt-get install clamav-daemon

def scan_for_viruses(file_path: str) -> bool:
    """Returns True if the file is clean, False if infected."""
    try:
        import pyclamd
        cd = pyclamd.ClamdUnixSocket()
        result = cd.scan_file(file_path)
        if result is None:
            return True  # clean
        # result is {path: ('FOUND', 'Virus.Name')} if infected
        return False
    except Exception:
        # If ClamAV is unavailable, fail closed (reject the file)
        return False

print("pyclamd: scan uploaded files against ClamAV before storing them.")
```

### SSRF Prevention (ASVS 12.6)

```python
render_diagram("assets/ssrf_prevention.png")
```

**SSRF (Server-Side Request Forgery)** tricks the server into making requests to internal resources. Prevent it by allowlisting outbound domains.

```python
from urllib.parse import urlparse
import requests

ALLOWED_FETCH_DOMAINS = {'api.trusted-partner.com', 'cdn.example.com'}

def safe_fetch(url: str) -> requests.Response:
    """Only fetch from allowlisted external domains."""
    parsed = urlparse(url)
    if parsed.hostname not in ALLOWED_FETCH_DOMAINS:
        raise ValueError(f"Domain {parsed.hostname!r} is not in the allowlist.")
    if parsed.scheme not in ('http', 'https'):
        raise ValueError("Only HTTP/HTTPS schemes are allowed.")
    return requests.get(url, timeout=5)

print("SSRF prevention: validate URL against domain allowlist before making server-side requests.")
```

---
## 11. API and Web Services (ASVS Ch.13)

APIs introduce unique security concerns: larger attack surface, missing browser protections, and increased risk of key leakage.

```python
render_diagram("assets/api_web_service_threats.png")
```

```python
render_diagram("assets/ch13_asvs_requirements.png")
```

### ASVS Ch.13 Key Requirements

| Section | Requirement |
|---------|-------------|
| **13.1.1** | No sensitive data in URL parameters; use POST body or headers |
| **13.1.3** | Consistent character encoding (UTF-8) across the API |
| **13.2.1** | Validate HTTP method against expected operations |
| **13.2.2** | Validate request body against a JSON schema |
| **13.2.3** | Use CSRF tokens for cookie-authenticated APIs |

### Threats to APIs and Web Services

- **Web Service**: XSS, CSRF, cookie sniffing
- **APIs**: increased risk of API key leakage, abuse of endpoints
- **Both**: SQL injection, DoS, insecure transport, error disclosure

### API Authentication with API Keys

```python
import secrets
import hashlib
from django.db import models
from functools import wraps
from django.http import JsonResponse

class APIKey(models.Model):
    user = models.ForeignKey('auth.User', on_delete=models.CASCADE)
    key_hash = models.CharField(max_length=64, unique=True)  # store hash, never plaintext
    created_at = models.DateTimeField(auto_now_add=True)
    is_active = models.BooleanField(default=True)

    @classmethod
    def generate(cls, user):
        raw_key = secrets.token_urlsafe(32)  # 256 bits of randomness
        key_hash = hashlib.sha256(raw_key.encode()).hexdigest()
        cls.objects.create(user=user, key_hash=key_hash)
        return raw_key  # return once to the user; never stored again

def api_key_required(view_func):
    @wraps(view_func)
    def wrapper(request, *args, **kwargs):
        raw_key = request.headers.get('X-API-Key', '')
        if not raw_key:
            return JsonResponse({'error': 'API key required'}, status=401)
        key_hash = hashlib.sha256(raw_key.encode()).hexdigest()
        try:
            api_key = APIKey.objects.get(key_hash=key_hash, is_active=True)
            request.api_user = api_key.user
        except APIKey.DoesNotExist:
            return JsonResponse({'error': 'Invalid API key'}, status=401)
        return view_func(request, *args, **kwargs)
    return wrapper

print("API key auth: store only SHA-256 hash; return key once; authenticate via X-API-Key header.")
```

### JSON Schema Validation (ASVS 13.2.2)

```python
# pip install jsonschema
import json
import jsonschema
from django.http import JsonResponse
from django.views.decorators.http import require_POST

TRANSFER_SCHEMA = {
    "type": "object",
    "required": ["amount", "recipient_iban"],
    "properties": {
        "amount": {"type": "number", "minimum": 0.01, "maximum": 10000.0},
        "recipient_iban": {"type": "string", "pattern": "^[A-Z]{2}[0-9]{2}[A-Z0-9]{4,30}$"},
        "reference": {"type": "string", "maxLength": 140},
    },
    "additionalProperties": False,  # reject unknown fields
}

@require_POST
@api_key_required
def transfer_api(request):
    try:
        body = json.loads(request.body)
    except json.JSONDecodeError:
        return JsonResponse({'error': 'Invalid JSON'}, status=400)
    
    try:
        jsonschema.validate(instance=body, schema=TRANSFER_SCHEMA)
    except jsonschema.ValidationError as e:
        return JsonResponse({'error': e.message}, status=400)
    
    # Process validated transfer...
    return JsonResponse({'status': 'accepted'})

print("JSON schema validation: enforce expected structure and types for all API inputs.")
```

### CORS Configuration for Django REST APIs

```python
# pip install django-cors-headers

# settings.py
CORS_SETTINGS_EXAMPLE = {
    # Only allow specific trusted origins (not '*')
    'CORS_ALLOWED_ORIGINS': [
        'https://app.example.com',
        'https://admin.example.com',
    ],
    # Allow credentials (cookies) only from trusted origins
    'CORS_ALLOW_CREDENTIALS': True,
    # Restrict allowed methods (ASVS 14.5.1)
    'CORS_ALLOW_METHODS': [
        'GET',
        'POST',
        'PUT',
        'PATCH',
        'DELETE',
    ],
}

print("CORS: use CORS_ALLOWED_ORIGINS with specific domains; never use '*' with credentials.")
```

---
## Summary

This module covered the full OWASP ASVS v4.0.3 Level 1 requirements as they apply to Django 4 applications.

| Module | ASVS Chapters | Key Django Tools |
|--------|--------------|------------------|
| Authentication | Ch.2 | `django.contrib.auth`, `django-ratelimit`, `django-otp` |
| Session Management | Ch.3 | `SESSION_COOKIE_*`, `session.cycle_key()`, step-up auth |
| Access Control | Ch.4 | `@permission_required`, `LoginRequiredMixin`, CSRF middleware |
| Validation & Encoding | Ch.5 | Django Forms, ORM parameterised queries, `escape()` |
| Data Protection | Ch.6, 8, 9 | Signed cookies, `SECURE_SSL_REDIRECT`, HSTS, `fernet-fields` |
| Logging | Ch.7 | Python `logging`, custom `Filter`, generic error pages |
| Config Hardening | Ch.10, 14 | `pip --require-hashes`, `safety`, security headers, `DEBUG=False` |
| Business Logic | Ch.11 | Workflow session state, per-user rate limiting |
| File Security | Ch.12 | Extension allowlist, randomised names, pyclamd, SSRF allowlist |
| API Security | Ch.13 | API key auth, jsonschema, `django-cors-headers` |

### Key Principle

> **Security is not one and done.** Keep up with new vulnerabilities, update dependencies regularly, and re-run ASVS verification as the application evolves.

### Further Resources
- [OWASP ASVS v4.0.3](https://owasp.org/www-project-application-security-verification-standard/)
- [OWASP Cheatsheets](https://cheatsheetseries.owasp.org/)
- [Django Security documentation](https://docs.djangoproject.com/en/4.2/topics/security/)
- `python manage.py check --deploy` — validate production settings